In [1]:
import sys
sys.path.append("../")
import neo4j
import pandas as pd
import pandas as pd
from utils_embeddings import load_embedding_model_std
from neo4j_graphrag.indexes import create_vector_index
from neo4j_graphrag.indexes import upsert_vectors
from neo4j_graphrag.types import EntityType
from qdrant_client import QdrantClient
from neo4j_graphrag.retrievers import QdrantNeo4jRetriever
import json

c:\Users\paul-\anaconda3\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NEO4J_URI = "bolt://localhost:7688"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "busticket123"

driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

In [3]:
model = load_embedding_model_std()

Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2


In [4]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [5]:
COLLECTION_NAME = "aoipoi_embeddings_std"

In [7]:
# 1. Verbindungen herstellen
client = QdrantClient("localhost", port=6333)

print("1. Verbindungen hergestellt")

# 8. Debug: Prüfen der gespeicherten Punkte in Qdrant
collection_info = client.get_collection(collection_name=COLLECTION_NAME)
print(f"\nQdrant Collection '{COLLECTION_NAME}' enthält {collection_info.points_count} Vektoren")


search_texts = [
    "Gemium Rahlstedt",
]
    

# Jede Suchanfrage testen
for search_text in search_texts:
    print(f"\nSuche nach: '{search_text}'")
    
    # Embedding erzeugen
    query_embedding = model.encode(search_text).tolist()
    
    # Suche durchführen
    try:

        # Direkter Check in Qdrant
        qdrant_results = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_embedding,
            limit=2
        )
        
        if qdrant_results:
            print("\nDirekte Qdrant-Ergebnisse:")
            for i, res in enumerate(qdrant_results, 1):
                print(f"Qdrant-Ergebnis {i}:")
                print(f"  ID: {res.id} (Typ: {type(res.id).__name__})")
                print(f"  Payload: {res.payload}")
                print(f"  Score: {res.score:.4f}")
                
                # Neo4j-Knoten direkt abfragen
                with driver.session() as session:
                    neo4j_id = res.payload.get("neo4j_id")
                    if neo4j_id:
                        neo4j_result = session.run(
                            "MATCH (p:POI {elementId: $id}) RETURN p.name AS name",
                            id=neo4j_id
                        )
                        record = neo4j_result.single()
                        if record:
                            print(f"  Entsprechender Neo4j-Knoten gefunden: {record['name']}")
                        else:
                            print(f"  KEIN Neo4j-Knoten mit ID={neo4j_id} gefunden!")
    
    except Exception as e:
        print(f"Fehler bei der Suche: {str(e)}")

driver.close()
print("Test abgeschlossen")

1. Verbindungen hergestellt

Qdrant Collection 'aoipoi_embeddings_std' enthält 11311 Vektoren

Suche nach: 'Gemium Rahlstedt'

Direkte Qdrant-Ergebnisse:
Qdrant-Ergebnis 1:
  ID: 194341c8-5fa5-4bd0-aad6-19f5425439a2 (Typ: str)
  Payload: {'neo4j_elementId': '4:30f9f47a-615f-4018-a291-0f284d9f1880:12031', 'name': 'Arthro-Clinic Rahlstedt', 'description': '', 'tags': [], 'label': 'POI', 'location': {'lon': 10.155575, 'lat': 53.604858}}
  Score: 0.6231
Qdrant-Ergebnis 2:
  ID: 30e78936-a5e8-4983-9df7-7320b2af8b0c (Typ: str)
  Payload: {'neo4j_elementId': '4:30f9f47a-615f-4018-a291-0f284d9f1880:11221', 'name': 'Rahlstedt - Kinderhaus', 'description': '', 'tags': [], 'label': 'POI', 'location': {'lon': 10.156799, 'lat': 53.600452}}
  Score: 0.5871
Test abgeschlossen


C:\Users\paul-\AppData\Local\Temp\ipykernel_11836\661588449.py:27: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_results = client.search(
C:\Users\paul-\AppData\Local\Temp\ipykernel_11836\661588449.py:42: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:
